In [1]:
from utils.imports import *

### 2.1.c)

In [233]:
# 1. Declaring the variables (e.g., upper and lower bounds of the random initialization and number of elements equivalent to the number of input neurons);
elements:int = 5
lwb, upb = 0, 10
random_vector= torch.tensor([rdm.randint(lwb,upb) for item in range(elements)], dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# 2.1 Building the 1DCN and Linearity;
cnv1 = nn.Conv1d(in_channels=1, out_channels=2, padding='same', bias=True, kernel_size=3)
linear1 = nn.Linear(in_features=5, out_features=10, bias=True)

# 2.2 Weight transfer;
# 2.2.1 Zero out all the randomly initialized weights within W matrix of linear1;
linear1.weight.data.zero_()

# 3. Weight transfer loop mechanism
with torch.no_grad():
    linear1.weight.data.zero_()
    linear1.bias.data.zero_()
    all_Kernels = cnv1.weight.data
    pad:int = 1
    # 2. Loop through the 10 output rows
    for record in range(10):
        # 3. Determine which filter to use and calculate the local step 's'
        if record >= 5:
            current_kernel = all_Kernels[1, 0] # Grab filter 1
            s = record - 5                     # s resets to 0-4 for the second half
            linear1.bias.data[record:] = cnv1.bias.data[1]
        else:
            current_kernel = all_Kernels[0, 0] # Grab filter 0
            s = record                         # s is just 0-4 for the first half
            linear1.bias.data[:record] = cnv1.bias.data[0]

        # 4. Loop through the 3 elements of our chosen kernel
        for k in range(len(current_kernel)):
            c = s+k-pad # assigning a particular value of the kernel denoted by `k` at a certain place in weight matrix labelled as `s`
            # 5. Check if the column index is within the matrix bounds (0 to 4)
            if c>=0 and c<5:
                # Place the actual kernel weight into the matrix
                linear1.weight.data[record,c]=current_kernel[k]

print("Final Matrix:")
print(linear1.weight.data)
print(linear1.bias.data)

# 5. Feed both the CN and LIN layers with randomly generated tensor vector;
print(cnv1(random_vector))
print(linear1(random_vector))




Final Matrix:
tensor([[-0.2845,  0.1860,  0.0000,  0.0000,  0.0000],
        [ 0.3137, -0.2845,  0.1860,  0.0000,  0.0000],
        [ 0.0000,  0.3137, -0.2845,  0.1860,  0.0000],
        [ 0.0000,  0.0000,  0.3137, -0.2845,  0.1860],
        [ 0.0000,  0.0000,  0.0000,  0.3137, -0.2845],
        [-0.3492, -0.4915,  0.0000,  0.0000,  0.0000],
        [-0.4172, -0.3492, -0.4915,  0.0000,  0.0000],
        [ 0.0000, -0.4172, -0.3492, -0.4915,  0.0000],
        [ 0.0000,  0.0000, -0.4172, -0.3492, -0.4915],
        [ 0.0000,  0.0000,  0.0000, -0.4172, -0.3492]])
tensor([ 0.0535,  0.0535,  0.0535,  0.0535,  0.0000, -0.0833, -0.0833, -0.0833,
        -0.0833, -0.0833])
tensor([[[-0.7123,  1.1114,  0.6700,  0.0135,  0.7101],
         [-2.4630, -3.4333, -3.0905, -2.4567, -1.6840]]],
       grad_fn=<ConvolutionBackward0>)
tensor([[[-0.7123,  1.1114,  0.6700,  0.0135,  0.6566, -2.4630, -3.4333,
          -3.0905, -2.4567, -1.6840]]], grad_fn=<ViewBackward0>)
